<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.
Требования к базовому классу PaymentMethod:
• Атрибуты: ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).
• Методы:
o ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.
o CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.
o GetPaymentDetails(): метод для получения деталей способа оплаты.
Требования к производным классам:
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

// ==========================================
// 1. ИНТЕРФЕЙСЫ
// ==========================================
public interface IRefundable
{
    bool Refund(decimal amount);
}

public interface ISecurePayment
{
    bool IsSecure { get; }
    string GetSecurityInfo();
}

public interface ILoggable
{
    void Log(string message);
}

// Новый интерфейс для явной реализации
public interface IAuditable
{
    void PerformAudit();
    string GetAuditTrail();
}

// ==========================================
// 2. БАЗОВЫЙ КЛАСС: PaymentMethod
// ==========================================
public class PaymentMethod : IRefundable, IAuditable
{
    // Исходные атрибуты
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    private string _currency;
    private bool _isActive;
    private string _description;
    private DateTime _createdDate;

    // 4 Новых атрибута
    private decimal _taxRate;
    private bool _isTaxIncluded;
    private string _receiptNumber;
    private string _departmentCode;

    // Свойства
    public int PaymentMethodId
    {
        get => _paymentMethodId;
        set => _paymentMethodId = value > 0 ? value : throw new ArgumentException("ID must be positive");
    }

    public string MethodName
    {
        get => _methodName;
        set => _methodName = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Name cannot be empty");
    }

    public decimal MinAmount
    {
        get => _minAmount;
        set => _minAmount = value >= 0 ? value : throw new ArgumentException("Min amount cannot be negative");
    }

    public string Currency
    {
        get => _currency;
        set => _currency = (!string.IsNullOrWhiteSpace(value) && value.Length == 3) ? value.ToUpper() : throw new ArgumentException("Currency must be 3 letters");
    }

    public bool IsActive { get => _isActive; set => _isActive = value; }
    public string Description { get => _description; set => _description = value ?? string.Empty; }
    public DateTime CreatedDate { get => _createdDate; private set => _createdDate = value; }

    public decimal TaxRate
    {
        get => _taxRate;
        set => _taxRate = (value >= 0 && value <= 1) ? value : throw new ArgumentException("Tax rate must be between 0 and 1");
    }

    public bool IsTaxIncluded { get => _isTaxIncluded; set => _isTaxIncluded = value; }
    public string ReceiptNumber { get => _receiptNumber; set => _receiptNumber = value ?? string.Empty; }
    public string DepartmentCode { get => _departmentCode; set => _departmentCode = value ?? string.Empty; }

    // Конструкторы
    public PaymentMethod()
    {
        PaymentMethodId = 1;
        MethodName = "Default";
        MinAmount = 0;
        Currency = "USD";
        IsActive = true;
        Description = "";
        CreatedDate = DateTime.Now;
        TaxRate = 0.20m;
        IsTaxIncluded = false;
        ReceiptNumber = "RCP-000000";
        DepartmentCode = "DEP-01";
    }

    public PaymentMethod(int id, string name, decimal minAmount) : this()
    {
        PaymentMethodId = id;
        MethodName = name;
        MinAmount = minAmount;
    }

    public PaymentMethod(int id, string name, decimal minAmount, string currency, string description) : this(id, name, minAmount)
    {
        Currency = currency;
        Description = description;
    }

    // Методы базового класса
    public virtual bool ProcessPayment(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine($"[{MethodName}] Payment method is inactive.");
            return false;
        }
        if (!CheckMinimumAmount(amount))
        {
            Console.WriteLine($"[{MethodName}] Amount {amount} {Currency} is below minimum {MinAmount} {Currency}.");
            return false;
        }
        Console.WriteLine($"[{MethodName}] Processed payment of {amount} {Currency}.");
        return


true;
    }

    public virtual bool ProcessPayment(decimal amount, string note)
    {
        Console.WriteLine($"[{MethodName}] Note: {note}");
        return ProcessPayment(amount);
    }

    public virtual bool CheckMinimumAmount(decimal amount) => amount >= MinAmount;
    public virtual bool CheckMinimumAmount(decimal amount, decimal fee) => (amount - fee) >= MinAmount;

    public virtual string GetPaymentDetails()
    {
        return $"ID: {PaymentMethodId}, Name: {MethodName}, MinAmount: {MinAmount} {Currency}, Active: {IsActive}, TaxRate: {TaxRate:P0}";
    }

    public virtual string GetPaymentDetails(bool includeDescription)
    {
        var details = GetPaymentDetails();
        if (includeDescription && !string.IsNullOrEmpty(Description))
            details += $", Description: {Description}";
        return details;
    }

    public void Activate() => IsActive = true;
    public void Deactivate() => IsActive = false;
    public void UpdateMinAmount(decimal newMin) => MinAmount = newMin;

    // 4 Новых метода базового класса
    public decimal CalculateTax(decimal amount) => IsTaxIncluded ? amount - (amount / (1 + TaxRate)) : amount * TaxRate;

    public string PrintReceipt(decimal amount)
    {
        decimal tax = CalculateTax(amount);
        return $"[Receipt #{ReceiptNumber}] Dept: {DepartmentCode} | Amount: {amount} {Currency} | Tax: {tax:N2} {Currency}";
    }

    public void SetTaxRate(decimal newRate, bool includeInPrice)
    {
        TaxRate = newRate;
        IsTaxIncluded = includeInPrice;
    }

    public bool ValidateTaxNumber(string taxNumber) => !string.IsNullOrWhiteSpace(taxNumber) && taxNumber.Length >= 10;

    public virtual bool Refund(decimal amount)
    {
        Console.WriteLine($"[{MethodName}] Refund of {amount} {Currency} processed (default).");
        return true;
    }

    // Явная реализация IAuditable
    void IAuditable.PerformAudit() => Console.WriteLine($"[Explicit Audit] Base PaymentMethod ID {PaymentMethodId} audited successfully.");
    string IAuditable.GetAuditTrail() => $"Audit Trail for {MethodName}: Created at {CreatedDate}, Status: {IsActive}";

    public bool CanProcessWith(PaymentMethod other, decimal amount) => this.CheckMinimumAmount(amount) && other.CheckMinimumAmount(amount);
}

// ==========================================
// 3. ПРОИЗВОДНЫЕ КЛАССЫ
// ==========================================

// --- OnlinePayment ---
public class OnlinePayment : PaymentMethod, ISecurePayment, IAuditable
{
    private string _paymentUrl;
    private string _providerName;
    private bool _isSecure;
    private string _securityToken;

    // 4 Новых атрибута
    private string _sslCertificateHash;
    private int _sessionTimeoutSeconds;
    private bool _isTwoFactorEnabled;
    private string _gatewayIpAddress;

    public string PaymentUrl { get => _paymentUrl; set => _paymentUrl = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("URL cannot be empty"); }
    public string ProviderName { get => _providerName; set => _providerName = value ?? string.Empty; }
    public bool IsSecure { get => _isSecure; set => _isSecure = value; }
    public string SecurityToken { get => _securityToken; set => _securityToken = value ?? string.Empty; }
    public string SslCertificateHash { get => _sslCertificateHash; set => _sslCertificateHash = value; }
    public int SessionTimeoutSeconds { get => _sessionTimeoutSeconds; set => _sessionTimeoutSeconds = value; }
    public bool IsTwoFactorEnabled { get => _isTwoFactorEnabled; set => _isTwoFactorEnabled = value; }
    public string GatewayIpAddress { get => _gatewayIpAddress; set => _gatewayIpAddress = value; }

    public OnlinePayment() : base()
    {
        PaymentUrl = "https://example.com/pay";
        ProviderName = "DefaultProvider";
        IsSecure = true;
        SecurityToken = Guid.NewGuid().ToString();
        SslCertificateHash = "SHA256_DEFAULT_HASH";
        SessionTimeoutSeconds = 300;
        IsTwoFactorEnabled = true;
        GatewayIpAddress = "192.168.1.1";
    }

    public OnlinePayment(int id, string name, decimal minAmount, string url, string provider) : base(id, name, minAmount)
    {
        PaymentUrl = url;
        ProviderName = provider;
        IsSecure = true;
        SecurityToken = Guid.NewGuid().ToString();
        SslCertificateHash = "SHA256_DEFAULT_HASH";
        SessionTimeoutSeconds = 300;
        IsTwoFactorEnabled = true;
        GatewayIpAddress = "192.168.1.1";
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount)) return false;
        Console.WriteLine($"  -> Redirecting to {PaymentUrl} for {amount} {Currency} via {ProviderName}.");
        Console.WriteLine($"  -> Security token: {SecurityToken}");
        return true;
    }

    public override string GetPaymentDetails() => base.GetPaymentDetails() + $", URL: {PaymentUrl}, Provider: {ProviderName}, Secure: {IsSecure}";

    public override bool Refund(decimal amount)
    {
        Console.WriteLine($"[{MethodName}] Online refund of {amount} {Currency} to {PaymentUrl} processed.");
        return true;
    }

    public string GetSecurityInfo() => $"Secure: {IsSecure}, Token: {SecurityToken}, IP: {GatewayIpAddress}";
    public string GeneratePaymentLink(decimal amount) => $"{PaymentUrl}?amount={amount}&currency={Currency}";

    // 4 Новых метода
    public void RenewSecurityToken() { SecurityToken = Guid.NewGuid().ToString(); Console.WriteLine($"[{MethodName}] Security token renewed."); }
    public bool VerifySslCertificate(string hash) => SslCertificateHash.Equals(hash, StringComparison.OrdinalIgnoreCase);
    public void ToggleTwoFactor(bool enable) { IsTwoFactorEnabled = enable; Console.WriteLine($"[{MethodName}] 2FA set to {enable}"); }
    public string ExportGatewayConfig() => $"Gateway: {ProviderName}, IP: {GatewayIpAddress}, Timeout: {SessionTimeoutSeconds}s";

    // Явная реализация IAuditable
    void IAuditable.PerformAudit() => Console.WriteLine($"[Explicit Audit] OnlinePayment gateway '{ProviderName}' security inspected.");
    string IAuditable.GetAuditTrail() => $"Online Audit: URL={PaymentUrl}, 2FA={IsTwoFactorEnabled}";
}

// --- MobilePayment (Многоуровневое наследование) ---
public class MobilePayment : OnlinePayment
{
    private string _phoneNumber;
    private string _operatorName;

    // 4 Новых атрибута
    private string _simCardId;
    private bool _isRoaming;
    private decimal _smsFee;
    private string _deviceOs;

    public string PhoneNumber { get => _phoneNumber; set => _phoneNumber = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Phone cannot be empty"); }
    public string OperatorName { get => _operatorName; set => _operatorName = value ?? string.Empty; }
    public string SimCardId { get => _simCardId; set => _simCardId = value; }
    public bool IsRoaming { get => _isRoaming; set => _isRoaming = value; }
    public decimal SmsFee { get => _smsFee; set => _smsFee = value; }
    public string DeviceOs { get => _deviceOs; set => _deviceOs = value; }

    public MobilePayment() : base()
    {
        PhoneNumber = "+1234567890";
        OperatorName = "DefaultOperator";
        SimCardId = "SIM-99999";
        IsRoaming = false;
        SmsFee = 0.05m;
        DeviceOs = "Android";
    }

    public MobilePayment(int id, string name, decimal minAmount, string url, string provider, string phone, string op) : base(id, name, minAmount, url, provider)
    {
        PhoneNumber = phone;
        OperatorName = op;
        SimCardId = "SIM-99999";
        IsRoaming = false;
        SmsFee = 0.05m;
        DeviceOs = "Android";
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount)) return false;
        Console.WriteLine($"  -> Mobile payment from {PhoneNumber} ({OperatorName}) confirmed.");
        return true;
    }

    public override string GetPaymentDetails() => base.GetPaymentDetails() + $", Phone: {PhoneNumber}, Operator: {OperatorName}, Roaming: {IsRoaming}";

    // 4 Новых метода
    public void ToggleRoaming(bool roam) { IsRoaming = roam; Console.WriteLine($"[{MethodName}] Roaming status changed to {roam}"); }
    public decimal CalculateTotalWithSms(decimal amount) => amount + SmsFee;
    public void UpdateDeviceOs(string newOs) { DeviceOs = newOs; Console.WriteLine($"[{MethodName}] Device OS updated to {newOs}"); }
    public string GetSimInfo() => $"SIM ID: {SimCardId}, Operator: {OperatorName}";
}

// --- BankTransfer ---
public class BankTransfer : PaymentMethod, IAuditable
{
    private string _bankData;
    private decimal _commissionRate;
    private string _bankName;
    private string _accountNumber;

    // 4 Новых атрибута
    private string _swiftCode;
    private string _correspondentAccount;
    private bool _isInstantTransfer;
    private decimal _dailyLimit;

    public string BankData { get => _bankData; set => _bankData = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Bank data cannot be empty"); }
    public decimal CommissionRate { get => _commissionRate; set => _commissionRate = (value >= 0 && value <= 1) ? value : throw new ArgumentException("Commission rate must be between 0 and 1"); }
    public string BankName { get => _bankName; set => _bankName = value ?? string.Empty; }
    public string AccountNumber { get => _accountNumber; set => _accountNumber = value ?? string.Empty; }
    public string SwiftCode { get => _swiftCode; set => _swiftCode = value ?? string.Empty; }
    public string CorrespondentAccount { get => _correspondentAccount; set => _correspondentAccount = value ?? string.Empty; }
    public bool IsInstantTransfer { get => _isInstantTransfer; set => _isInstantTransfer = value; }
    public decimal DailyLimit { get => _dailyLimit; set => _dailyLimit = value; }

    public BankTransfer() : base()
    {
        BankData = "DefaultBankData";
        CommissionRate = 0.01m;
        BankName = "DefaultBank";
        AccountNumber = "0000000000";
        SwiftCode = "DEFAULTSWIFT";
        CorrespondentAccount = "30101810400000000225";
        IsInstantTransfer = false;
        DailyLimit = 50000m;
    }

    public BankTransfer(int id, string name, decimal minAmount, string bankData, decimal commissionRate, string bankName, string account) : base(id, name, minAmount)
    {
        BankData = bankData;
        CommissionRate = commissionRate;
        BankName = bankName;
        AccountNumber = account;
        SwiftCode = "DEFAULTSWIFT";
        CorrespondentAccount = "30101810400000000225";
        IsInstantTransfer = false;
        DailyLimit = 50000m;
    }

    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal commission = amount * CommissionRate;
        decimal effectiveAmount = amount - commission;
        bool result = effectiveAmount >= MinAmount;
        Console.WriteLine($"[{MethodName}] Amount: {amount}, Commission ({CommissionRate:P0}): {commission}, Effective: {effectiveAmount}, Min: {MinAmount} -> {(result ? "OK" : "Below minimum")}");
        return result;
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (!IsActive) { Console.WriteLine($"[{MethodName}] Payment method is inactive."); return false; }
        if (amount > DailyLimit) { Console.WriteLine($"[{MethodName}] Amount exceeds daily limit of {DailyLimit} {Currency}."); return false; }
        if (!CheckMinimumAmount(amount)) { Console.WriteLine($"[{MethodName}] Payment amount after commission is below minimum."); return false; }
        decimal commission = amount * CommissionRate;
        Console.WriteLine($"[{MethodName}] Bank transfer of {amount} {Currency} to {BankName} (Account: {AccountNumber}) processed.");
        Console.WriteLine($"  -> Commission: {commission} {Currency}, Net: {amount - commission} {Currency}");
        return true;
    }

    public override string GetPaymentDetails() => base.GetPaymentDetails() + $", Bank: {BankName}, Account: {AccountNumber}, SWIFT: {SwiftCode}";
    public decimal CalculateCommission(decimal amount) => amount * CommissionRate;

    // 4 Новых метода
    public void UpdateDailyLimit(decimal newLimit) { DailyLimit = newLimit; Console.WriteLine($"[{MethodName}] Daily limit updated to {newLimit} {Currency}"); }
    public bool VerifySwiftCode(string swift) => SwiftCode.Equals(swift, StringComparison.OrdinalIgnoreCase);
    public void ToggleInstantTransfer(bool instant) { IsInstantTransfer = instant; Console.WriteLine($"[{MethodName}] Instant transfer set to {instant}"); }
    public string GetBankRequisites() => $"Bank: {BankName}, Acc: {AccountNumber}, Corr: {CorrespondentAccount}, SWIFT: {SwiftCode}";

    // Явная реализация IAuditable
    void IAuditable.PerformAudit() => Console.WriteLine($"[Explicit Audit] BankTransfer compliance checked for SWIFT: {SwiftCode}");
    string IAuditable.GetAuditTrail() => $"Bank Audit: Bank={BankName}, DailyLimit={DailyLimit}";
}

// --- CashPayment ---
public class CashPayment : PaymentMethod, ILoggable
{
    private string _cashPickupPoint;
    private bool _isAvailable;
    private string _pickupAddress;
    private string _workingHours;

    // 4 Новых атрибута
    private bool _isCashAccepted;
    private decimal _maxCashLimit;
    private string _cashierName;
    private int _safeBoxNumber;

    public string CashPickupPoint { get => _cashPickupPoint; set => _cashPickupPoint = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Pickup point cannot be empty"); }
    public bool IsAvailable { get => _isAvailable; set => _isAvailable = value; }
    public string PickupAddress { get => _pickupAddress; set => _pickupAddress = value ?? string.Empty; }
    public string WorkingHours { get => _workingHours; set => _workingHours = value ?? string.Empty; }
    public bool IsCashAccepted { get => _isCashAccepted; set => _isCashAccepted = value; }
    public decimal MaxCashLimit { get => _maxCashLimit; set => _maxCashLimit = value; }
    public string CashierName { get => _cashierName; set => _cashierName = value ?? string.Empty; }
    public int SafeBoxNumber { get => _safeBoxNumber; set => _safeBoxNumber = value; }

    public CashPayment() : base()
    {
        CashPickupPoint = "Main Office";
        IsAvailable = true;
        PickupAddress = "123 Main St";
        WorkingHours = "9:00-18:00";
        IsCashAccepted = true;
        MaxCashLimit = 10000m;
        CashierName = "John Doe";
        SafeBoxNumber = 5;
    }

    public CashPayment(int id, string name, decimal minAmount, string pickupPoint, string address, string hours) : base(id, name, minAmount)
    {
        CashPickupPoint = pickupPoint;
        IsAvailable = true;
        PickupAddress = address;
        WorkingHours = hours;
        IsCashAccepted = true;
        MaxCashLimit = 10000m;
        CashierName = "John Doe";
        SafeBoxNumber = 5;
    }

    public override string GetPaymentDetails() => base.GetPaymentDetails() + $", PickupPoint: {CashPickupPoint}, Address: {PickupAddress}, Hours: {WorkingHours}";

    public override bool ProcessPayment(decimal amount)
    {
        if (!IsAvailable || !IsCashAccepted) { Console.WriteLine($"[{MethodName}] Cash point {CashPickupPoint} cannot accept cash right now."); return false; }
        if (amount > MaxCashLimit) { Console.WriteLine($"[{MethodName}] Amount exceeds maximum cash limit of {MaxCashLimit}."); return false; }
        return base.ProcessPayment(amount);
    }

    public override bool Refund(decimal amount)
    {
        Console.WriteLine($"[{MethodName}] Cash refund of {amount} {Currency} at {CashPickupPoint} processed by {CashierName}.");
        return true;
    }

    public void Log(string message) => Console.WriteLine($"[LOG][{MethodName}] {message}");

    // 4 Новых метода
    public void AssignCashier(string name) { CashierName = name; Console.WriteLine($"[{MethodName}] Cashier changed to {name}"); }
    public void UpdateMaxLimit(decimal limit) { MaxCashLimit = limit; Console.WriteLine($"[{MethodName}] Max cash limit updated to {limit}"); }
    public void ToggleCashAcceptance(bool accept) { IsCashAccepted = accept; Console.WriteLine($"[{MethodName}] Cash acceptance status: {accept}"); }
    public string GetSafeStatus() => $"Safe Box #{SafeBoxNumber} at {CashPickupPoint}, Managed by {CashierName}";
}

// ==========================================
// 4. УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ И ДЖЕНЕРИКИ
// ==========================================

// Сервис с внедрением зависимостей (Dependency Injection)
public class PaymentProcessor
{
    private readonly ILoggable _logger;
    private readonly IRefundable _refundablePayment;

    public PaymentProcessor(ILoggable logger, IRefundable refundablePayment)
    {
        _logger = logger ?? throw new ArgumentNullException(nameof(logger));
        _refundablePayment = refundablePayment ?? throw new ArgumentNullException(nameof(refundablePayment));
    }

    public void ExecuteRefundFlow(decimal amount)
    {
        _logger.Log($"Starting refund process for amount: {amount}");
        bool success = _refundablePayment.Refund(amount);
        if (success) _logger.Log("Refund completed successfully.");
        else _logger.Log("Refund failed.");
    }
}

// Коллекция
public class PaymentCollection<T> where T : PaymentMethod
{
    private List<T> _items = new List<T>();

    public void Add(T item) => _items.Add(item);
    public void Remove(T item) => _items.Remove(item);
    public int Count => _items.Count;
    public T this[int index] => _items[index];
    public IEnumerable<T> Items => _items;

    public void ProcessAll(decimal amount)
    {
        foreach (var item in _items)
        {
            item.ProcessPayment(amount);
            Console.WriteLine();
        }
    }

    public T FindById(int id) => _items.FirstOrDefault(x => x.PaymentMethodId == id);
    public decimal GetTotalMinAmount() => _items.Sum(x => x.MinAmount);

    public void DisplayAll()
    {
        foreach (var item in _items)
        {
            Console.WriteLine(item.GetPaymentDetails());
            Console.WriteLine();
        }
    }
}

// ==========================================
// 5. ТОЧКА ВХОДА (MAIN)
// ==========================================
class Program
{
    static void Main()
    {
        Console.WriteLine("=== 1. Демонстрация новых методов и атрибутов ===");
        var basePayment = new PaymentMethod(1, "Base Card", 10m);
        Console.WriteLine(basePayment.PrintReceipt(150m));
        Console.WriteLine($"Calculated tax: {basePayment.CalculateTax(150m)}");
        Console.WriteLine();

        Console.WriteLine("=== 2. Явная реализация интерфейса IAuditable ===");
        IAuditable auditableBase = basePayment;
        auditableBase.PerformAudit();
        Console.WriteLine(auditableBase.GetAuditTrail());

        var online = new OnlinePayment(2, "Online Card", 20m, "https://pay.example.com", "SecurePay");
        IAuditable auditableOnline = online;
        auditableOnline.PerformAudit();
        Console.WriteLine(auditableOnline.GetAuditTrail());
        Console.WriteLine();

        Console.WriteLine("=== 3. Управление Зависимостями (Dependency Injection) ===");
        var cash = new CashPayment(5, "Cash Payment", 1m, "Office #1", "Lenina 10", "10:00-20:00");
        PaymentProcessor processor = new PaymentProcessor(cash, cash);
        processor.ExecuteRefundFlow(250m);
        Console.WriteLine();

        Console.WriteLine("=== 4. Работа с коллекцией PaymentCollection ===");
        var collection = new PaymentCollection<PaymentMethod>();
        collection.Add(basePayment);
        collection.Add(online);
        collection.Add(cash);
        
        Console.WriteLine($"Всего элементов в коллекции: {collection.Count}");
        Console.WriteLine($"Суммарная минимальная сумма: {collection.GetTotalMinAmount()}");
    }
}